# 05a · Export the property graph for Neo4j bulk import
Writes node and relationship CSVs in the `neo4j-admin database import` format from the silver layer, following the Assignment 1 graph model:

- `(:User {userId, flagged, flaggedSessions, maxScore})` – `flagged` comes from the K-Means anomaly flags of notebook 03a.
- `(:Product {productId, categoryId, categoryCode, brand, lastPrice})`, `(:Category)`, `(:Brand)`
- `(:User)-[:INTERACTED {type, count, firstSeen, lastSeen}]->(:Product)` — one relationship per (user, product, type), `type` ∈ view/cart/purchase.
- `(:Product)-[:IN_CATEGORY]->(:Category)`, `(:Product)-[:OF_BRAND]->(:Brand)`

Files land in `data/neo4j-import/`. Sessions are not nodes (same decision as Assignment 1).

In [1]:
import os, json, time, shutil
from pyspark.sql import SparkSession, functions as F

DATA = "/home/jovyan/work/data"; GOLD = f"{DATA}/gold"
OUT = "/home/jovyan/work/ml/outputs"; IMP = f"{DATA}/neo4j-import"
shutil.rmtree(IMP, ignore_errors=True); os.makedirs(IMP)

spark = (SparkSession.builder.master("local[*]").appName("05a_export_graph")
         .config("spark.driver.memory", "12g").config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.session.timeZone", "UTC").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
ev = spark.read.parquet(f"{DATA}/silver/events")

def write(df, name, parts):
    (df.coalesce(parts).write.mode("overwrite").option("header", False).option("escape", '"').csv(f"{IMP}/{name}"))

def header(name, cols):
    open(f"{IMP}/{name}_header.csv", "w").write(",".join(cols) + "\n")
stats = {}

## Nodes

In [2]:
t0 = time.time()
susp = spark.read.parquet(f"{GOLD}/suspicious_users")
users = (ev.select("user_id").distinct()
         .join(susp, "user_id", "left")
         .select(F.col("user_id").alias("userId"),
                 F.col("flagged_sessions").isNotNull().alias("flagged"),
                 F.coalesce("flagged_sessions", F.lit(0)).cast("int").alias("flaggedSessions"),
                 F.col("max_score").cast("float").alias("maxScore")))
write(users, "users", 4)
header("users", ["userId:ID(User)", "flagged:boolean", "flaggedSessions:int", "maxScore:float"])

latest = (ev.groupBy("product_id").agg(
            F.max_by("category_id", "event_time").alias("categoryId"),
            F.max_by("category_code", "event_time").alias("categoryCode"),
            F.max_by("brand", "event_time").alias("brand"),
            F.max_by("price", "event_time").alias("lastPrice")))
products = latest.select(F.col("product_id").alias("productId"), "categoryId", "categoryCode", "brand", F.col("lastPrice").cast("double"))
write(products, "products", 1)
header("products", ["productId:ID(Product)", "categoryId:long", "categoryCode", "brand", "lastPrice:double"])

cats = latest.filter("categoryId is not null").groupBy("categoryId").agg(F.first("categoryCode").alias("code"))
write(cats, "categories", 1); header("categories", ["categoryId:ID(Category)", "code"])
brands = latest.filter("brand is not null").select(F.col("brand").alias("name")).distinct()
write(brands, "brands", 1); header("brands", ["name:ID(Brand)"])

write(latest.filter("categoryId is not null").select("product_id", "categoryId"), "in_category", 1)
header("in_category", [":START_ID(Product)", ":END_ID(Category)"])
write(latest.filter("brand is not null").select("product_id", "brand"), "of_brand", 1)
header("of_brand", [":START_ID(Product)", ":END_ID(Brand)"])
stats["nodes_s"] = round(time.time() - t0, 1)

## Relationships
`INTERACTED` per (user, product, type) with count and first/last timestamp (epoch seconds). This is the full graph: all 61 days, including the test week used in notebook 04.

In [3]:
t0 = time.time()
inter = (ev.groupBy("user_id", "product_id", "event_type")
           .agg(F.count("*").cast("int").alias("count"),
                F.min(F.col("event_time").cast("long")).alias("firstSeen"),
                F.max(F.col("event_time").cast("long")).alias("lastSeen"))
           .select("user_id", "product_id", "event_type", "count", "firstSeen", "lastSeen"))
write(inter, "interacted", 16)
header("interacted", [":START_ID(User)", ":END_ID(Product)", "type", "count:int", "firstSeen:long", "lastSeen:long"])
stats["interacted_s"] = round(time.time() - t0, 1)

In [4]:
# verify counts straight from the written files
def n(name): return spark.read.csv(f"{IMP}/{name}").count()
stats.update({k: n(k) for k in ["users", "products", "categories", "brands", "interacted", "in_category", "of_brand"]})
stats["flagged_users"] = spark.read.csv(f"{IMP}/users").filter("_c1 = 'true'").count()
json.dump(stats, open(f"{OUT}/05a_metrics.json", "w"), indent=2)
print(stats)
spark.stop()

{'nodes_s': 31.2, 'interacted_s': 15.4, 'users': 5315274, 'products': 205415, 'categories': 689, 'brands': 4287, 'interacted': 59553973, 'in_category': 205415, 'of_brand': 158233, 'flagged_users': 101532}
